# Rananiti Data Cleaning and Quality Review

Check required fields, duplicates, numeric parsing, and feed-specific extras encodings without modifying the source CSVs.

All inputs are loaded from `data/raw/`. Outputs are saved to `outputs/`.

In [1]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(min(8, os.cpu_count() or 2)))
ROOT_DIR=Path.cwd()
if not (ROOT_DIR/"data"/"raw"/"ball_by_ball.csv").exists(): ROOT_DIR=ROOT_DIR.parent
sys.path.insert(0,str(ROOT_DIR))
from config import BALLS_FILE, MATCHES_FILE, OUTPUTS
OUTPUTS.mkdir(parents=True,exist_ok=True)
balls=pd.read_csv(BALLS_FILE,low_memory=False)
matches=pd.read_csv(MATCHES_FILE,low_memory=False) if MATCHES_FILE.exists() else pd.DataFrame()
print("Project:",ROOT_DIR,"| ball rows:",len(balls),"| matches:",matches.match_id.nunique() if not matches.empty else "metadata missing")
from src.data_processing.clean_data import clean_ball_data
clean=clean_ball_data(balls)
required=["match_id","innings","overid","over_ball_num","striker_id","bowler_id","runs"]
quality={"source_rows":int(len(balls)),"rows_after_required_field_cleaning":int(len(clean)),"removed_rows":int(len(balls)-len(clean)),"duplicate_rows":int(clean.duplicated().sum()),"missing_required_fields":{c:int(balls[c].isna().sum()) for c in required if c in balls}}
if "extras1_runs" in clean and "extras_type" in clean:
    quality["extras1_runs_type_counts"]=clean.groupby(["extras_type","extras1_runs"],dropna=False).size().reset_index(name="rows").to_dict(orient="records")
print(json.dumps(quality,indent=2,default=str))
(OUTPUTS/"data_quality_report.json").write_text(json.dumps(quality,indent=2,default=str),encoding="utf-8")
print("Saved quality report:",OUTPUTS/"data_quality_report.json")


Project: C:\Users\SHEETAL\OneDrive\ドキュメント\Desktop\Rananiti | ball rows: 200999 | matches: 883


{
  "source_rows": 200999,
  "rows_after_required_field_cleaning": 200999,
  "removed_rows": 0,
  "duplicate_rows": 0,
  "missing_required_fields": {
    "match_id": 0,
    "innings": 0,
    "overid": 0,
    "over_ball_num": 0,
    "striker_id": 0,
    "bowler_id": 0,
    "runs": 0
  },
  "extras1_runs_type_counts": [
    {
      "extras_type": "Overstep",
      "extras1_runs": 1,
      "rows": 576
    },
    {
      "extras_type": "Penalty",
      "extras1_runs": 7,
      "rows": 1
    },
    {
      "extras_type": "byes",
      "extras1_runs": 4,
      "rows": 529
    },
    {
      "extras_type": "legbyes",
      "extras1_runs": 3,
      "rows": 2748
    },
    {
      "extras_type": "other",
      "extras1_runs": 5,
      "rows": 264
    },
    {
      "extras_type": "wide",
      "extras1_runs": 2,
      "rows": 7714
    },
    {
      "extras_type": NaN,
      "extras1_runs": 6,
      "rows": 189167
    }
  ]
}
Saved quality report: C:\Users\SHEETAL\OneDrive\ドキュメント\Desktop\Ranani